# 08. Review-text-grouped versus movie-disjoint evaluation

This notebook compares the repository's conventional review-text-grouped split with a stricter movie-disjoint split. The same feature blocks, model search space, random seed, preprocessing rules, and evaluation metrics are used in both settings.

The conventional split keeps identical review text together but may contain reviews from the same movie in both partitions. The movie-disjoint split keeps movies, and movies connected by duplicate review text, in one partition. Model selection uses only each setting's training partition; the corresponding test partition is evaluated only after fitting the selected model.


## 1. Setup

The default search below is deliberately small enough for a reproducible comparison. Increase `SAMPLE_SIZE`, `MODEL_GRID`, or `TFIDF_GRID` only when computational resources allow it. The comparison remains fair because both split strategies use the same settings.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start Jupyter from the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.clean_spoilers import clean_text
from src.data_cleaning import clean_data
from src.logistic_regression import refit_logistic_regression, tune_logistic_regression
from src.metrics import evaluate
from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import create_train_test_split

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "split_comparison"

RANDOM_STATE = 67
N_FOLDS = 5
GROUPED_SPLIT = "review_text"
MOVIE_SPLIT = "movie_id"
FEATURE_COLUMNS = INITIAL_FEATURES.copy()

# Selection uses a reproducible subset; final refitting uses every training row.
SAMPLE_SIZE = 10_000
OVERSAMPLE_POSITIVE = True
MAX_ITER = 1_000
TOL = 1e-3

MODEL_GRID = {"penalty": ["l2"], "C": [0.1, 1.0]}
TFIDF_GRID = {"ngram_range": [(1, 1), (1, 2)], "max_features": [5_000]}
print(f"Project root: {PROJECT_ROOT}")
print(f"Feature blocks: {FEATURE_COLUMNS}")


Project root: c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers
Feature blocks: ['review_text', 'review_summary', 'rating', 'plot_summary', 'plot_synopsis', 'genre', 'duration', 'days_since_release']


## 2. Generate both outer splits

`create_train_test_split` is deterministic for the configured random seed. It checks that duplicate review text does not cross the partitions and, for the movie-disjoint setting, that movie IDs do not cross the partitions.


In [2]:
clean_data()
_, grouped_train, grouped_test = create_train_test_split(GROUPED_SPLIT)
_, movie_train, movie_test = create_train_test_split(MOVIE_SPLIT)

split_frames = {
    "review_text_grouped": (grouped_train, grouped_test, GROUPED_SPLIT),
    "movie_disjoint": (movie_train, movie_test, MOVIE_SPLIT),
}
print("Generated and loaded both split strategies.")


Loading raw data...
Cleaning movie IDs...
Cleaning complete.
Cleaned reviews saved to: C:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\cleaned\IMDB_reviews.json
Cleaned movie details saved to: C:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\cleaned\IMDB_movie_details.json
Train: 459,130 (80.00%), spoiler rate 26.30%
Test:  114,783 (20.00%), spoiler rate 26.30%
Train: 459,361 (80.04%), spoiler rate 26.31%
Test:  114,552 (19.96%), spoiler rate 26.26%
Generated and loaded both split strategies.


## 3. Validate split properties

The review-text-grouped split is the repository's conventional benchmark. Movie overlap is expected there. The movie-disjoint split must have zero movie overlap. Review-text overlap is checked after spoiler-cue removal because the feature pipeline uses the same normalisation when forming groups.


In [3]:
def split_summary(name, train, test):
    train_movies = set(train["movie_id"].dropna())
    test_movies = set(test["movie_id"].dropna())
    train_text = set(train["review_text"].map(clean_text))
    test_text = set(test["review_text"].map(clean_text))
    return {
        "split": name,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_spoiler_rate": train["is_spoiler"].mean(),
        "test_spoiler_rate": test["is_spoiler"].mean(),
        "train_movies": len(train_movies),
        "test_movies": len(test_movies),
        "movie_overlap": len(train_movies & test_movies),
        "cue_removed_text_overlap": len(train_text & test_text),
    }

split_checks = pd.DataFrame([
    split_summary(name, train, test)
    for name, (train, test, _) in split_frames.items()
])
display(split_checks)

assert split_checks.loc[split_checks["split"] == "movie_disjoint", "movie_overlap"].iat[0] == 0
assert (split_checks["cue_removed_text_overlap"] == 0).all()
assert (split_checks["train_rows"] > 0).all() and (split_checks["test_rows"] > 0).all()


,split,train_rows,test_rows,train_spoiler_rate,test_spoiler_rate,train_movies,test_movies,movie_overlap,cue_removed_text_overlap
0,review_text_grouped,459130,114783,0.262973,0.262974,1572,1570,1570,0
1,movie_disjoint,459361,114552,0.263074,0.262571,1272,300,0,0


## 4. Prepare review and movie features

The split files contain review fields and `movie_id`; plot fields and other movie metadata are joined from the cleaned movie-details file. Labels and grouping IDs remain separate from model features.


In [4]:
movies = pd.read_json(
    CLEANED_DIR / "IMDB_movie_details.json",
    lines=True,
    convert_dates=False,
)

prepared_splits = {}
for name, (train_reviews, test_reviews, group_by) in split_frames.items():
    train_features = prepare_features(train_reviews, movies)
    test_features = prepare_features(test_reviews, movies)
    train_data = train_features.copy()
    test_data = test_features.copy()
    train_data["is_spoiler"] = train_reviews["is_spoiler"].to_numpy()
    test_data["is_spoiler"] = test_reviews["is_spoiler"].to_numpy()
    train_data["movie_id"] = train_reviews["movie_id"].to_numpy()
    test_data["movie_id"] = test_reviews["movie_id"].to_numpy()
    prepared_splits[name] = (train_data, test_data, group_by)

print("Prepared feature columns:", list(prepared_splits["review_text_grouped"][0].columns))


C:\Users\quanh\AppData\Local\Temp\ipykernel_20768\2989970778.py:1: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  movies = pd.read_json(


Prepared feature columns: ['review_text', 'review_summary', 'rating', 'plot_summary', 'plot_synopsis', 'genre', 'duration', 'days_since_release', 'is_spoiler', 'movie_id']


## 5. Tune and fit one identical logistic-regression protocol per split

Each split is tuned independently using the same candidate grid. Candidate ranking uses mean validation macro-F1, and only converged candidates are eligible. The final model is refit on all rows in that split's training partition.


In [5]:
def select_sample(data, sample_size, random_state):
    if sample_size is None or sample_size >= len(data):
        return data.reset_index(drop=True)
    rng = np.random.default_rng(random_state)
    indices = np.sort(rng.choice(len(data), size=sample_size, replace=False))
    return data.iloc[indices].reset_index(drop=True)

experiment_results = {}
for name, (full_train, test_data, group_by) in prepared_splits.items():
    tuning_data = select_sample(full_train, SAMPLE_SIZE, RANDOM_STATE)
    summary, fold_scores, best = tune_logistic_regression(
        tuning_data,
        MODEL_GRID,
        TFIDF_GRID,
        feature_columns=FEATURE_COLUMNS,
        k=N_FOLDS,
        group_by=group_by,
        random_state=RANDOM_STATE,
        oversample_positive=OVERSAMPLE_POSITIVE,
        max_iter=MAX_ITER,
        tol=TOL,
        verbose=True,
    )
    if best is None:
        raise RuntimeError(f"No converged model was found for {name}.")

    model = refit_logistic_regression(
        full_train,
        best,
        feature_columns=FEATURE_COLUMNS,
        random_state=RANDOM_STATE,
        oversample_positive=OVERSAMPLE_POSITIVE,
        max_iter=MAX_ITER,
        tol=TOL,
    )
    probabilities = model.predict_proba(test_data)[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    metrics = evaluate(test_data["is_spoiler"], predictions, probabilities)
    metrics["average_precision"] = metrics.pop("pr_auc")
    metrics.update({
        "split": name,
        "group_by": group_by,
        "best_penalty": best["model"]["penalty"],
        "best_C": best["model"]["C"],
        "best_ngram_range": str(best["tfidf"]["ngram_range"]),
        "tuning_rows": len(tuning_data),
        "train_rows": len(full_train),
        "test_rows": len(test_data),
        "test_movies": test_data["movie_id"].nunique(),
    })
    experiment_results[name] = {
        "model": model,
        "best": best,
        "summary": summary,
        "fold_scores": fold_scores,
        "metrics": metrics,
        "test_data": test_data,
        "probabilities": probabilities,
        "predictions": predictions,
    }
    print(f"Completed {name}: macro-F1={metrics['macro_f1']:.4f}")


TF-IDF (1, 1), fold 1/5, {'C': 0.1, 'penalty': 'l2'}, 8 features: F1=0.4636, macro F1=0.6098, converged=True
TF-IDF (1, 1), fold 1/5, {'C': 1.0, 'penalty': 'l2'}, 8 features: F1=0.4264, macro F1=0.6015, converged=True
TF-IDF (1, 1), fold 2/5, {'C': 0.1, 'penalty': 'l2'}, 8 features: F1=0.5032, macro F1=0.6395, converged=True
TF-IDF (1, 1), fold 2/5, {'C': 1.0, 'penalty': 'l2'}, 8 features: F1=0.4840, macro F1=0.6370, converged=True
TF-IDF (1, 1), fold 3/5, {'C': 0.1, 'penalty': 'l2'}, 8 features: F1=0.4889, macro F1=0.6325, converged=True
TF-IDF (1, 1), fold 3/5, {'C': 1.0, 'penalty': 'l2'}, 8 features: F1=0.4604, macro F1=0.6297, converged=True
TF-IDF (1, 1), fold 4/5, {'C': 0.1, 'penalty': 'l2'}, 8 features: F1=0.4779, macro F1=0.6207, converged=True
TF-IDF (1, 1), fold 4/5, {'C': 1.0, 'penalty': 'l2'}, 8 features: F1=0.4641, macro F1=0.6250, converged=True
TF-IDF (1, 1), fold 5/5, {'C': 0.1, 'penalty': 'l2'}, 8 features: F1=0.4725, macro F1=0.6130, converged=True
TF-IDF (1, 1), fold

## 6. Compare held-out test performance

The difference below is descriptive: the movie-disjoint test set answers a different generalisation question from the review-text-grouped test set. It should not be presented as a paired statistical test because the test examples differ.


In [6]:
comparison = pd.DataFrame([
    result["metrics"] for result in experiment_results.values()
]).set_index("split")
metric_columns = [
    "accuracy", "precision", "recall", "f1", "macro_f1", "average_precision",
    "tn", "fp", "fn", "tp",
]
display(comparison[[column for column in metric_columns if column in comparison]])

if {"review_text_grouped", "movie_disjoint"} <= set(comparison.index):
    gap = (
        comparison.loc["movie_disjoint", metric_columns]
        - comparison.loc["review_text_grouped", metric_columns]
    )
    display(gap.rename("movie_disjoint_minus_review_text_grouped").to_frame())


,accuracy,precision,recall,f1,macro_f1,average_precision,tn,fp,fn,tp
split,,,,,,,,,,
review_text_grouped,0.717650,0.474870,0.69614,0.564600,0.677843,0.584678,61361,23237,9172,21013
movie_disjoint,0.718119,0.472189,0.62431,0.537697,0.667472,0.539375,63484,20990,11300,18778


,movie_disjoint_minus_review_text_grouped
accuracy,0.000469
precision,-0.002681
recall,-0.071830
f1,-0.026903
macro_f1,-0.010371
average_precision,-0.045303
tn,2123.000000
fp,-2247.000000
fn,2128.000000
tp,-2235.000000


## 7. Inspect model evidence and per-movie behaviour

The first table checks whether the model's strongest coefficients are concentrated in review or plot blocks. The second table gives a per-movie view of movie-disjoint errors. Per-movie scores can be unstable for movies with very few test reviews, so include review counts when interpreting them.


In [11]:
def get_column_transformer_feature_names(transformer):
    """Return feature names for a fitted ColumnTransformer."""
    feature_names = []

    for name, fitted_transformer, columns in transformer.transformers_:
        if fitted_transformer == "drop":
            continue

        if fitted_transformer == "passthrough":
            branch_names = [str(column) for column in columns]
        elif hasattr(fitted_transformer, "named_steps"):
            # Text branches end with a vectorizer; use it because the
            # preceding FunctionTransformer has no feature-name API.
            final_transformer = fitted_transformer.steps[-1][1]
            branch_names = final_transformer.get_feature_names_out()
        else:
            branch_names = fitted_transformer.get_feature_names_out(columns)

        feature_names.extend(
            f"{name}__{branch_name}" for branch_name in branch_names
        )

    return np.asarray(feature_names, dtype=object)


for name, result in experiment_results.items():
    classifier = result["model"].named_steps["classifier"]
    transformer = result["model"].named_steps["features"]
    if hasattr(classifier, "coef_"):
        feature_names = get_column_transformer_feature_names(transformer)
        coefficients = classifier.coef_.ravel()
        if len(feature_names) != len(coefficients):
            raise ValueError(
                f"Feature-name count ({len(feature_names)}) does not match "
                f"coefficient count ({len(coefficients)}) for {name}."
            )
        top_positive = np.argsort(coefficients)[-15:][::-1]
        top_negative = np.argsort(coefficients)[:15]
        print(f"\n{name}: strongest positive coefficients")
        display(pd.DataFrame({"feature": feature_names[top_positive], "coefficient": coefficients[top_positive]}))
        print(f"{name}: strongest negative coefficients")
        display(pd.DataFrame({"feature": feature_names[top_negative], "coefficient": coefficients[top_negative]}))



review_text_grouped: strongest positive coefficients


,feature,coefficient
0,review_text__spoilers,10.907381
1,review_text__spoiler,10.778171
2,review_summary__spoilers,5.790761
3,review_summary__spoiler,4.376210
4,review_text__scene,3.800865
5,review_text__killed,3.625918
6,review_text__ending,3.569535
7,days_since_release__x0,3.420313
8,review_text__end,3.401654
9,review_text__death,3.207639


review_text_grouped: strongest negative coefficients


,feature,coefficient
0,review_summary__vidiot,-1.760413
1,review_summary__nutshell,-1.686298
2,review_text__script,-1.637604
3,plot_summary__edu,-1.629460
4,review_text__role,-1.576136
5,review_text__movies,-1.437677
6,review_text__director,-1.415021
7,review_text__performance,-1.381680
8,review_text__direction,-1.316727
9,review_text__action,-1.303294



movie_disjoint: strongest positive coefficients


,feature,coefficient
0,review_text__spoiler,8.717692
1,review_text__spoilers,8.328410
2,review_summary__spoilers,6.448145
3,review_summary__spoiler,4.348303
4,review_text__scene,3.867355
5,review_text__ending,3.692400
6,review_text__end,3.133764
7,review_text__killed,3.124872
8,days_since_release__x0,3.056554
9,review_text__the end,2.830287


movie_disjoint: strongest negative coefficients


,feature,coefficient
0,review_text__performance,-1.869152
1,review_text__role,-1.786077
2,review_text__movies,-1.647154
3,review_text__director,-1.592746
4,review_text__script,-1.493595
5,review_text__action,-1.443095
6,plot_summary__edu,-1.341427
7,review_text__actors,-1.255098
8,review_text__screen,-1.252896
9,review_text__work,-1.236533


In [12]:
movie_result = experiment_results["movie_disjoint"]
movie_test = movie_result["test_data"].copy()
movie_test["prediction"] = movie_result["predictions"]
movie_test["probability"] = movie_result["probabilities"]
movie_test["correct"] = movie_test["prediction"] == movie_test["is_spoiler"]

per_movie = movie_test.groupby("movie_id").apply(
    lambda group: pd.Series({
        "reviews": len(group),
        "spoiler_rate": group["is_spoiler"].mean(),
        "accuracy": group["correct"].mean(),
        "predicted_spoiler_rate": group["prediction"].mean(),
    }),
    include_groups=False,
).sort_values(["reviews", "accuracy"], ascending=[False, True])
display(per_movie.head(20))


,reviews,spoiler_rate,accuracy,predicted_spoiler_rate
movie_id,,,,
tt0468569,4845.0,0.305882,0.712487,0.312693
tt0111161,4361.0,0.219216,0.730337,0.296721
tt0167260,2729.0,0.179553,0.829974,0.189080
tt0137523,2480.0,0.185081,0.818548,0.178629
tt0068646,2137.0,0.196537,0.658868,0.416940
tt0110912,1994.0,0.199097,0.799398,0.161986
tt0050083,916.0,0.271834,0.763100,0.146288
tt1446714,739.0,0.541272,0.748309,0.581867
tt0903624,735.0,0.352381,0.687075,0.344218


## 8. Save reproducible comparison outputs

The source split files remain under `data/splits/`. The exported CSVs contain experiment summaries and predictions; they do not replace the source data.


In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.reset_index().to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
split_checks.to_csv(OUTPUT_DIR / "split_checks.csv", index=False)
per_movie.reset_index().to_csv(OUTPUT_DIR / "movie_disjoint_per_movie.csv", index=False)

for name, result in experiment_results.items():
    predictions = result["test_data"][["movie_id", "is_spoiler"]].copy()
    predictions["prediction"] = result["predictions"]
    predictions["probability"] = result["probabilities"]
    predictions.to_csv(OUTPUT_DIR / f"{name}_predictions.csv", index=False)

print(f"Saved comparison outputs to {OUTPUT_DIR}")


Saved comparison outputs to c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\outputs\split_comparison


## 9. Discussions

### Split validity and comparability

This notebook compares two evaluation settings using the same feature set, logistic-regression model family, hyperparameter grid, random seed, grouped validation procedure, TF-IDF configuration, oversampling policy, and evaluation metrics. Both strategies preserve the approximately 26.3% spoiler rate in their training and test partitions. The review-text-grouped split contains 459,130 training reviews and 114,783 test reviews, while the movie-disjoint split contains 459,361 training reviews and 114,552 test reviews.

The validation checks confirm the intended distinction between the protocols. The review-text-grouped split keeps identical review text together but allows movie overlap: 1,570 movies occur in both partitions. The movie-disjoint split has 1,272 training movies and 300 test movies, with zero movie overlap. Cue-removed review text also has zero overlap across either pair of partitions, so the ablation does not introduce text leakage.

### Predictive performance

The review-text-grouped model achieved accuracy of 0.7177, positive-class F1 of 0.5646, macro-F1 of 0.6778, and average precision of 0.5847. The movie-disjoint model achieved accuracy of 0.7181, positive-class F1 of 0.5377, macro-F1 of 0.6675, and average precision of 0.5394.

Accuracy was almost unchanged, but the stricter movie-disjoint evaluation reduced positive-class recall from 0.6961 to 0.6243, positive-class F1 by approximately 0.0269, macro-F1 by approximately 0.0104, and average precision by approximately 0.0453. Accuracy alone therefore hides a meaningful reduction in spoiler detection when the model must generalise to unseen movies.

### What the model appears to learn

The coefficient inspection shows that explicit warning terms such as `spoiler` and `spoilers` are among the strongest positive predictors in both settings. Other positive predictors include terms associated with endings, death, scenes, and characters being killed. Negative coefficients are dominated by general review vocabulary such as `performance`, `director`, `script`, `acting`, and `movies`, which is more indicative of ordinary film commentary than spoiler disclosure.

These coefficients support the motivation for checking shortcut learning: some predictive power comes from obvious spoiler-warning language rather than only from semantic evidence of plot revelation.

### Per-movie behaviour and limitations

The per-movie results show substantial variation in accuracy and predicted spoiler rate across unseen movies. Per-movie estimates should be interpreted alongside review counts because results for movies with few test reviews are unstable.

The comparison uses a deliberately small model-selection search and a 10,000-row tuning sample, although the selected model is refit on the complete training partition. The results should therefore be treated as evidence about the effect of split strategy, not as a final model benchmark. Future analysis should report uncertainty, investigate class-specific metrics in more detail, and use movie-disjoint results when the research question concerns generalisation to previously unseen movies.
